In [2]:
# Define matrix A
A = [
    [7,9,8], 
    [7,8,1], 
    [1,2,6], 
]

# Define vector B
B = [7,8,7]


class Gauss(): 
    def __init__(self): 
        self.found_max = None
        self.found_row = None 
        self.found_column = None
        
    def print(self, *matrices): 
        for matrix in matrices: 
            print("[") 
            for row in matrix:
                if isinstance(row, list):
                    print("\t", [round(float(x), 4) for x in row])
                else:
                    print("\t", round(float(row), 4))
            print("]") 

    def display(self, matrixA, matrixB, matrixC, title=""):
        if title:
            print(f"> {title}")
        header = "          " + "        ".join(f"{var}" for var in matrixC)
        print(header)
        print("-" * (len(header) + 10))
        
        for i in range(len(matrixA)):
            row_content = "  ".join(f"{val:8.4f}" for val in matrixA[i])
            print(f"|  {row_content}  |  =  {matrixB[i]:8.4f}")
        print("\n")

    def get_ordinal(self, n):
        if 11 <= (n % 100) <= 13:
            suffix = 'th'
        else:
            suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
        return f"{n}{suffix}"

    def reorganized(self, C, X):
        paired = list(zip(C, X))
        paired.sort(key=lambda x: int(x[0][1:])) 
        result_C, result_X = zip(*paired)
        return list(result_C), list(result_X)

    def dimension_1D(self, matrix): 
        return len(matrix)
            
    def swap_element(self, vector, index_A, index_B): 
        vector[index_A], vector[index_B] = vector[index_B], vector[index_A]
        return vector

    def swap_row(self, matrixA, matrixB, index_A, index_B):
        matrixA[index_A], matrixA[index_B] = matrixA[index_B], matrixA[index_A]
        matrixB = self.swap_element(matrixB, index_A, index_B)
        return [matrixA, matrixB]

    def swap_column(self, matrixA, matrixC, index_A, index_B): 
        for i_row in range(len(matrixA)):
            matrixA[i_row][index_A], matrixA[i_row][index_B] = matrixA[i_row][index_B], matrixA[i_row][index_A]
        matrixC = self.swap_element(matrixC, index_A, index_B)
        return [matrixA, matrixC]
    
    def obtain_max(self, matrix, step):    
        l_max = -1
        l_row = step
        l_col = step
        for r in range(step, len(matrix)):
            for c in range(step, len(matrix)):
                if abs(matrix[r][c]) > l_max:
                    l_max = abs(matrix[r][c])
                    l_row = r
                    l_col = c
        self.found_max = l_max
        self.found_row = l_row
        self.found_column = l_col

    def row_pivot_normalize(self, matrixA, matrixB, i_step): 
        l_pivot = matrixA[i_step][i_step]
        if abs(l_pivot) > 1e-12:
            for c in range(i_step, len(matrixA)):
                matrixA[i_step][c] /= l_pivot
            matrixB[i_step] /= l_pivot
        return [matrixA, matrixB]

    def column_pivot_zero(self, matrixA, matrixB, i_step):
        for i_row in range(i_step + 1, len(matrixA)):
            l_factor = matrixA[i_row][i_step]
            for i_col in range(i_step, len(matrixA)):
                matrixA[i_row][i_col] -= l_factor * matrixA[i_step][i_col]
            matrixB[i_row] -= l_factor * matrixB[i_step]
        return [matrixA, matrixB]

    def back_substitution(self, matrixA, matrixB, matrixC):
        n = len(matrixA)
        results = [0] * n
        print("> Back Substitution:")
        
        for i in range(n - 1, -1, -1):
            total_c = 0
            process = f"{matrixC[i]} = ({matrixB[i]:.4f}"
            for j in range(i + 1, n):
                weighted_val = matrixA[i][j] * results[j]
                total_c += weighted_val
                process += f" - [{matrixA[i][j]:.4f} * {results[j]:.4f}]"
            
            raw_val = (matrixB[i] - total_c) / matrixA[i][i]
            results[i] = raw_val if abs(raw_val) > 1e-12 else 0.0
            process += f") = {results[i]:.4f}"
            print(process)
        print("\n")
        return results

        
gauss = Gauss()
dimension = gauss.dimension_1D(A) 
C = [f"X{n}" for n in range(1, dimension + 1)]

is_singular = False 

for i_row in range(dimension): 
    print(f" {'-'*24} {gauss.get_ordinal(i_row + 1)} Pivot {'-'*24}")
    gauss.display(A, B, C, "Display Initial")
    gauss.obtain_max(A, i_row) 

    if abs(gauss.found_max) < 1e-12:
        print("\n[!] ERROR: Matrix is singular or has no unique solution.")
        is_singular = True
        break 

    A, B = gauss.swap_row(A, B, i_row, gauss.found_row)
    gauss.display(A, B, C, "After Row Swap")
    
    A, C = gauss.swap_column(A, C, i_row, gauss.found_column)
    gauss.display(A, B, C, "After Column Swap")

    A, B = gauss.row_pivot_normalize(A, B, i_row)
    gauss.display(A, B, C, "After Normalizing")
    
    A, B = gauss.column_pivot_zero(A, B, i_row)
    gauss.display(A, B, C, "After Zeroing Below")
    
if not is_singular:
    X = gauss.back_substitution(A, B, C)
    C, X = gauss.reorganized(C, X)

    print("FINAL SOLUTIONS:")
    for (key, value) in list(zip(C,X)): 
        print(f"{key} = {value:8.4f}")
else:
    print("Execution stopped: Cannot compute back substitution on a singular matrix.")

 ------------------------ 1st Pivot ------------------------
> Display Initial
          X1        X2        X3
------------------------------------------
|    7.0000    9.0000    8.0000  |  =    7.0000
|    7.0000    8.0000    1.0000  |  =    8.0000
|    1.0000    2.0000    6.0000  |  =    7.0000


> After Row Swap
          X1        X2        X3
------------------------------------------
|    7.0000    9.0000    8.0000  |  =    7.0000
|    7.0000    8.0000    1.0000  |  =    8.0000
|    1.0000    2.0000    6.0000  |  =    7.0000


> After Column Swap
          X2        X1        X3
------------------------------------------
|    9.0000    7.0000    8.0000  |  =    7.0000
|    8.0000    7.0000    1.0000  |  =    8.0000
|    2.0000    1.0000    6.0000  |  =    7.0000


> After Normalizing
          X2        X1        X3
------------------------------------------
|    1.0000    0.7778    0.8889  |  =    0.7778
|    8.0000    7.0000    1.0000  |  =    8.0000
|    2.0000    1.0000    6